In [1]:
!pip install underthesea seqeval datasets transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 37.2 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=018fa315c5586e6dbc243d194931f564a0d6a8e06645033d22bd9a58942a85f2
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [2]:
!kaggle kernels output nguynlmhuy/nlp-assignment-1 -p .

Output file downloaded to ./input/raw_contracts.txt
Output file downloaded to ./output/chunks.txt
Output file downloaded to ./output/clauses.txt
Output file downloaded to ./output/dependency.json
Kernel log downloaded to ./nlp-assignment-1.log 


In [3]:
!kaggle kernels output nguynlmhuy/nlp-assignment2-1 -p .

raw_contracts.txt: Skipping, found more recently modified local copy (use --force to force download)
nlp-assignment-1.log: Skipping, found more recently modified local copy (use --force to force download)
chunks.txt: Skipping, found more recently modified local copy (use --force to force download)
clauses.txt: Skipping, found more recently modified local copy (use --force to force download)
dependency.json: Skipping, found more recently modified local copy (use --force to force download)
Output file downloaded to ./output/ner_results.json
Output file downloaded to ./phobert-ner-legal/added_tokens.json
Output file downloaded to ./phobert-ner-legal/bpe.codes
Output file downloaded to ./phobert-ner-legal/checkpoint-300/added_tokens.json
Output file downloaded to ./phobert-ner-legal/checkpoint-300/bpe.codes
Output file downloaded to ./phobert-ner-legal/checkpoint-300/config.json
Output file downloaded to ./phobert-ner-legal/checkpoint-300/model.safetensors
Output file downloaded to ./phobe

In [4]:
import os, json, re, random, time
import numpy as np
from pathlib import Path
from collections import Counter, defaultdict

OUTPUT_DIR      = "/kaggle/working/output"
CLAUSE_FILE     = f"{OUTPUT_DIR}/clauses.txt"
NER_FILE        = f"{OUTPUT_DIR}/ner_results.json"
SRL_LABEL_FILE  = "/kaggle/input/datasets/nguynlmhuy/srl-data/labeled_srl_data.json"
SRL_OUTPUT      = f"{OUTPUT_DIR}/srl_results.json"
MODEL_SAVE_DIR  = "/kaggle/working/vilegalbert-srl"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✓ Paths configured")
print(f"  Clauses         : {CLAUSE_FILE}")
print(f"  NER results     : {NER_FILE}")
print(f"  SRL labels      : {SRL_LABEL_FILE}")
print(f"  SRL output      : {SRL_OUTPUT}")
print(f"  Model save dir  : {MODEL_SAVE_DIR}")

✓ Paths configured
  Clauses         : /kaggle/working/output/clauses.txt
  NER results     : /kaggle/working/output/ner_results.json
  SRL labels      : /kaggle/input/datasets/nguynlmhuy/srl-data/labeled_srl_data.json
  SRL output      : /kaggle/working/output/srl_results.json
  Model save dir  : /kaggle/working/vilegalbert-srl


In [5]:
with open(CLAUSE_FILE, "r", encoding="utf-8") as f:
    all_clauses = [line.strip() for line in f if line.strip()]
print(f"✓ Loaded {len(all_clauses):,} clauses")

# Load NER results from Task 2.1
with open(NER_FILE, "r", encoding="utf-8") as f:
    ner_results = json.load(f)

ner_by_clause = {r["clause"]: r["entities"] for r in ner_results}
print(f"✓ Loaded {len(ner_results):,} NER results")
print(f"  With ≥1 entity : {sum(1 for r in ner_results if r['entities']):,}")

# Sample
for r in ner_results[:50]:
    if r["entities"]:
        print(f"\nSample:")
        print(f"  Clause  : {r['clause']}")
        for e in r["entities"]:
            print(f"  [{e['label']}] {e['text']}")
        break

✓ Loaded 1,147,999 clauses
✓ Loaded 1,216 NER results
  With ≥1 entity : 0


In [6]:
ROLES = ["Agent", "Predicate", "Theme", "Recipient", "Time", "Condition"]

LABEL_LIST = ["O"]
for role in ROLES:
    LABEL_LIST.append(f"B-{role}")
    LABEL_LIST.append(f"I-{role}")

LABEL2ID = {lbl: i for i, lbl in enumerate(LABEL_LIST)}
ID2LABEL  = {i: lbl for lbl, i in LABEL2ID.items()}

print(f"✓ Label scheme ({len(LABEL_LIST)} labels):")
for lbl, idx in LABEL2ID.items():
    print(f"  {idx:>2}  {lbl}")

✓ Label scheme (13 labels):
   0  O
   1  B-Agent
   2  I-Agent
   3  B-Predicate
   4  I-Predicate
   5  B-Theme
   6  I-Theme
   7  B-Recipient
   8  I-Recipient
   9  B-Time
  10  I-Time
  11  B-Condition
  12  I-Condition


In [7]:
# ================================================================
# This cell shows the EXACT format for labeled_srl_data.json
# Annotate ~300 clauses and save to OUTPUT_DIR/labeled_srl_data.json
# ================================================================

ANNOTATION_EXAMPLE = [
    {
        "clause": "Bên A phải bàn giao vật tư cho bên B.",
        "tokens": ["Bên", "A", "phải", "bàn", "giao", "vật", "tư", "cho", "bên", "B", "."],
        "srl_tags": [
            "B-Agent",      # Bên
            "I-Agent",      # A
            "O",            # phải
            "B-Predicate",  # bàn
            "I-Predicate",  # giao
            "B-Theme",      # vật
            "I-Theme",      # tư
            "O",            # cho
            "B-Recipient",  # bên
            "I-Recipient",  # B
            "O"             # .
        ]
    },
    {
        "clause": "Bên B phải thanh toán 10.000.000 VNĐ trước ngày 05/5/2024.",
        "tokens": ["Bên", "B", "phải", "thanh", "toán", "10.000.000", "VNĐ", "trước", "ngày", "05/5/2024", "."],
        "srl_tags": [
            "B-Agent",      # Bên
            "I-Agent",      # B
            "O",            # phải
            "B-Predicate",  # thanh
            "I-Predicate",  # toán
            "B-Theme",      # 10.000.000
            "I-Theme",      # VNĐ
            "B-Time",       # trước
            "I-Time",       # ngày
            "I-Time",       # 05/5/2024
            "O"             # .
        ]
    },
    {
        "clause": "Nếu thanh toán trễ hạn, mức phạt 1% mỗi ngày sẽ được áp dụng.",
        "tokens": ["Nếu", "thanh", "toán", "trễ", "hạn", ",", "mức", "phạt", "1%", "mỗi", "ngày", "sẽ", "được", "áp", "dụng", "."],
        "srl_tags": [
            "B-Condition",  # Nếu
            "I-Condition",  # thanh
            "I-Condition",  # toán
            "I-Condition",  # trễ
            "I-Condition",  # hạn
            "O",            # ,
            "O",            # mức
            "B-Predicate",  # phạt
            "B-Theme",      # 1%
            "B-Time",       # mỗi
            "I-Time",       # ngày
            "O",            # sẽ
            "O",            # được
            "B-Predicate",  # áp
            "I-Predicate",  # dụng
            "O"             # .
        ]
    }
]

print("✓ Annotation format example ready")
print(f"\nValid labels: {LABEL_LIST}")
print(f"\nRules:")
print(f"  1. tokens   : word_tokenize(clause) or clause.split()")
print(f"  2. srl_tags : BIO tags, len must == len(tokens)")
print(f"  3. Every clause must have ≥1 Predicate tag")
print(f"\nSave 300 annotated clauses to:")
print(f"  {SRL_LABEL_FILE}")

✓ Annotation format example ready

Valid labels: ['O', 'B-Agent', 'I-Agent', 'B-Predicate', 'I-Predicate', 'B-Theme', 'I-Theme', 'B-Recipient', 'I-Recipient', 'B-Time', 'I-Time', 'B-Condition', 'I-Condition']

Rules:
  1. tokens   : word_tokenize(clause) or clause.split()
  2. srl_tags : BIO tags, len must == len(tokens)
  3. Every clause must have ≥1 Predicate tag

Save 300 annotated clauses to:
  /kaggle/input/datasets/nguynlmhuy/srl-data/labeled_srl_data.json


In [8]:
# Run after annotation is complete
import copy
from collections import Counter

with open(SRL_LABEL_FILE, "r", encoding="utf-8") as f:
    labeled_data = json.load(f)

print(f"✓ Loaded {len(labeled_data)} annotated examples")

errors = []
for i, ex in enumerate(labeled_data):
    for key in ["clause", "tokens", "srl_tags"]:
        if key not in ex:
            errors.append(f"  Example {i}: missing '{key}'")
    if len(ex.get("tokens", [])) != len(ex.get("srl_tags", [])):
        errors.append(f"  Example {i}: length mismatch "
                      f"tokens={len(ex.get('tokens',[]))} "
                      f"srl_tags={len(ex.get('srl_tags',[]))}")
    for tag in ex.get("srl_tags", []):
        if tag not in LABEL2ID:
            errors.append(f"  Example {i}: unknown tag '{tag}'")
    if not any(t.startswith("B-Predicate") for t in ex.get("srl_tags", [])):
        errors.append(f"  Example {i}: no Predicate tag found")

if errors:
    print(f"\n✗ {len(errors)} errors:")
    for e in errors[:20]:
        print(e)
    raise ValueError("Fix annotation errors before continuing")
else:
    print("✓ All examples valid!")

tag_counts = Counter()
for ex in labeled_data:
    for tag in ex["srl_tags"]:
        if tag != "O":
            tag_counts[tag.replace("B-","").replace("I-","")] += 1

print("\nRole distribution (original):")
for role, cnt in tag_counts.most_common():
    print(f"  {role:<12} {cnt:>5}")

# ── Data augmentation: duplicate rare-role examples ──────────────
# Time + Agent + Recipient are underrepresented → oversample them
RARE_ROLES     = {"Time", "Agent", "Recipient"}
AUGMENT_TIMES  = 4    # repeat rare examples 4x

def has_role(ex, roles):
    return any(
        tag[2:] in roles
        for tag in ex["srl_tags"]
        if tag.startswith("B-")
    )

augmented = copy.deepcopy(labeled_data)
for ex in labeled_data:
    if has_role(ex, RARE_ROLES):
        for _ in range(AUGMENT_TIMES - 1):
            augmented.append(copy.deepcopy(ex))

print(f"\n✓ After augmentation: {len(augmented)} examples")
tag_counts_aug = Counter()
for ex in augmented:
    for tag in ex["srl_tags"]:
        if tag.startswith("B-"):
            tag_counts_aug[tag[2:]] += 1

print("Role distribution (augmented):")
for role, cnt in tag_counts_aug.most_common():
    bar = "█" * (cnt // 2)
    print(f"  {role:<12} {cnt:>5}  {bar}")

labeled_data = augmented

✓ Loaded 281 annotated examples
✓ All examples valid!

Role distribution (original):
  Theme         2023
  Condition     1020
  Predicate      704
  Recipient      281
  Agent          215
  Time            85

✓ After augmentation: 530 examples
Role distribution (augmented):
  Theme         1015  ███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  Predicate      942  ██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████

In [9]:
import copy

random.seed(42)

# Stratified split — ensure each role appears in both train and test
from collections import defaultdict

def get_roles(ex):
    return frozenset(
        tag[2:] for tag in ex["srl_tags"] if tag.startswith("B-")
    )

# Group by role combination
role_buckets = defaultdict(list)
for ex in labeled_data:
    key = get_roles(ex)
    role_buckets[key].append(ex)

train_data, test_data = [], []
random.seed(42)

for key, bucket in role_buckets.items():
    random.shuffle(bucket)
    split = max(1, int(len(bucket) * 0.85))
    train_data.extend(bucket[:split])
    test_data.extend(bucket[split:])

random.shuffle(train_data)
random.shuffle(test_data)

# Verify no overlap
train_keys = set(" ".join(ex["tokens"]) for ex in train_data)
test_keys  = set(" ".join(ex["tokens"]) for ex in test_data)
overlap    = train_keys & test_keys

print(f"✓ Train : {len(train_data)} examples")
print(f"✓ Test  : {len(test_data)} examples")
print(f"✓ Overlap: {len(overlap)} {'✓ clean' if not overlap else '⚠ LEAK!'}")

# Role coverage check
def role_coverage(data, name):
    roles = Counter()
    for ex in data:
        for tag in ex["srl_tags"]:
            if tag.startswith("B-"):
                roles[tag[2:]] += 1
    print(f"\n  {name} role coverage:")
    for role in ROLES:
        cnt = roles.get(role, 0)
        print(f"    {role:<12} {cnt:>4}")

role_coverage(train_data, "Train")
role_coverage(test_data,  "Test")

✓ Train : 442 examples
✓ Test  : 88 examples
✓ Overlap: 46 ⚠ LEAK!

  Train role coverage:
    Agent         155
    Predicate     784
    Theme         854
    Recipient     229
    Time           52
    Condition     244

  Test role coverage:
    Agent          33
    Predicate     158
    Theme         161
    Recipient      47
    Time           12
    Condition      52


In [10]:
import copy

random.seed(42)

all_labeled = copy.copy(labeled_data)
random.shuffle(all_labeled)

# Split FIRST before any sampling
split     = int(len(all_labeled) * 0.85)
train_data = all_labeled[:split]
test_data  = all_labeled[split:]

# Verify no overlap
train_keys = set(" ".join(ex["tokens"]) for ex in train_data)
test_keys  = set(" ".join(ex["tokens"]) for ex in test_data)
overlap    = train_keys & test_keys

print(f"✓ Train : {len(train_data)} examples")
print(f"✓ Test  : {len(test_data)} examples")
print(f"✓ Overlap: {len(overlap)} examples {'✓ clean' if not overlap else '⚠ LEAK!'}")

✓ Train : 450 examples
✓ Test  : 80 examples
✓ Overlap: 44 examples ⚠ LEAK!


In [11]:
from datasets import Dataset, DatasetDict

def to_hf_format(examples):
    out = {"tokens": [], "srl_tags": []}
    for ex in examples:
        out["tokens"].append(ex["tokens"])
        out["srl_tags"].append([LABEL2ID.get(t, 0) for t in ex["srl_tags"]])
    return out

ds = DatasetDict({
    "train": Dataset.from_dict(to_hf_format(train_data)),
    "test" : Dataset.from_dict(to_hf_format(test_data)),
})

print("✓ DatasetDict:")
print(ds)
print("\nSample:")
ex = ds["train"][0]
for tok, tag_id in zip(ex["tokens"][:8], ex["srl_tags"][:8]):
    print(f"  {tok:<20}  {ID2LABEL[tag_id]}")

✓ DatasetDict:
DatasetDict({
    train: Dataset({
        features: ['tokens', 'srl_tags'],
        num_rows: 450
    })
    test: Dataset({
        features: ['tokens', 'srl_tags'],
        num_rows: 80
    })
})

Sample:
  Đảm_bảo               B-Predicate
  các                   B-Theme
  điều_kiện             I-Theme
  hoạt_động             I-Theme
  của                   I-Theme
  Ban                   I-Theme
  Chỉ_đạo               I-Theme
  ,                     O


In [12]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("your_hf_token_here")
login(token)

In [13]:
from transformers import AutoTokenizer

MODEL_NAME = "ntphuc149/ViLegalBERT"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

print(f"✓ Tokenizer: {MODEL_NAME}")
print(f"  Fast       : {tokenizer.is_fast}")
print(f"  Vocab size : {tokenizer.vocab_size:,}")

sample = "Bên A phải bàn giao vật tư cho bên B."
enc = tokenizer(sample.split(), is_split_into_words=True)
print(f"\nTest: '{sample}'")
print(f"  Subwords: {tokenizer.convert_ids_to_tokens(enc['input_ids'])}")

config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

✓ Tokenizer: ntphuc149/ViLegalBERT
  Fast       : False
  Vocab size : 64,000

Test: 'Bên A phải bàn giao vật tư cho bên B.'
  Subwords: ['<s>', 'Bên', 'A', 'phải', 'bàn', 'giao', 'vật', 'tư', 'cho', 'bên', 'B.', '</s>']


In [14]:
MAX_SEQ_LEN = 256

def tokenize_and_align_labels(examples):
    all_input_ids      = []
    all_attention_mask = []
    all_labels         = []

    for words, orig_labels in zip(examples["tokens"], examples["srl_tags"]):
        pieces_per_word = [
            len(tokenizer.tokenize(w)) or 1
            for w in words
        ]
        enc = tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            max_length=MAX_SEQ_LEN,
            padding=False,
        )
        input_ids      = enc["input_ids"]
        attention_mask = enc["attention_mask"]

        labels       = [-100]
        token_budget = MAX_SEQ_LEN - 2
        used         = 0

        for word_idx, n_pieces in enumerate(pieces_per_word):
            if used + n_pieces > token_budget:
                break
            word_label = orig_labels[word_idx] if word_idx < len(orig_labels) else 0
            labels.append(word_label)
            labels.extend([-100] * (n_pieces - 1))
            used += n_pieces

        labels.append(-100)

        assert len(labels) == len(input_ids), (
            f"Mismatch: labels={len(labels)}, tokens={len(input_ids)}"
        )

        all_input_ids.append(input_ids)
        all_attention_mask.append(attention_mask)
        all_labels.append(labels)

    return {
        "input_ids"      : all_input_ids,
        "attention_mask" : all_attention_mask,
        "labels"         : all_labels,
    }

print("Tokenizing...")
tokenized_ds = ds.map(
    tokenize_and_align_labels,
    batched=True,
    batch_size=64,
    remove_columns=ds["train"].column_names,
    desc="Tokenizing",
)
print("\n✓ Tokenized:")
print(tokenized_ds)

Tokenizing...


Tokenizing:   0%|          | 0/450 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/80 [00:00<?, ? examples/s]


✓ Tokenized:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 450
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 80
    })
})


In [15]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels              = len(LABEL_LIST),
    id2label                = ID2LABEL,
    label2id                = LABEL2ID,
    ignore_mismatched_sizes = True,
)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✓ Model: {MODEL_NAME}")
print(f"  Total params    : {total/1e6:.1f}M")
print(f"  Trainable params: {trainable/1e6:.1f}M")
print(f"  Num labels      : {len(LABEL_LIST)}")

model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: ntphuc149/ViLegalBERT
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ Model: ntphuc149/ViLegalBERT
  Total params    : 134.4M
  Trainable params: 134.4M
  Num labels      : 13


In [16]:
import numpy as np
from seqeval.metrics import classification_report

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions    = np.argmax(logits, axis=-1)

    true_labels, true_preds = [], []
    for pred_seq, label_seq in zip(predictions, labels):
        true_labels.append([ID2LABEL[l] for p, l in zip(pred_seq, label_seq) if l != -100])
        true_preds.append( [ID2LABEL[p] for p, l in zip(pred_seq, label_seq) if l != -100])

    report = classification_report(true_labels, true_preds, output_dict=True)
    return {
        "f1"       : report.get("micro avg", {}).get("f1-score",  0.0),
        "precision": report.get("micro avg", {}).get("precision", 0.0),
        "recall"   : report.get("micro avg", {}).get("recall",    0.0),
    }

print("✓ Metrics defined")

✓ Metrics defined


In [17]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
from transformers import EarlyStoppingCallback

args = TrainingArguments(
    output_dir                  = MODEL_SAVE_DIR,
    num_train_epochs            = 40,            
    per_device_eval_batch_size  = 16,
    learning_rate               = 2e-5,         
    lr_scheduler_type           = "cosine",      
    warmup_ratio                = 0.1,          
    weight_decay                = 0.01,          
    label_smoothing_factor      = 0.0,           
    eval_strategy               = "epoch",       
    logging_strategy            = "epoch",
    save_strategy               = "epoch",
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",          
    greater_is_better           = True,
    report_to                   = "none",
    fp16                        = True,
    push_to_hub                 = False,
    ddp_find_unused_parameters  = False,
    seed                        = 42,
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model            = model,
    args             = args,
    train_dataset    = tokenized_ds["train"],
    eval_dataset     = tokenized_ds["test"],
    processing_class = tokenizer,
    data_collator    = data_collator,
    compute_metrics  = compute_metrics,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=8)],  # more patience
)

eff_batch = args.per_device_train_batch_size * 2   # T4x2
steps     = (len(tokenized_ds["train"]) / eff_batch) * args.num_train_epochs
print("✓ Trainer ready")
print(f"  Train examples   : {len(tokenized_ds['train'])}")
print(f"  Test examples    : {len(tokenized_ds['test'])}")
print(f"  Effective batch  : {eff_batch}")
print(f"  Total steps est. : {steps:.0f}")
print(f"  Est. time        : ~{steps/8/60:.1f} min")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✓ Trainer ready
  Train examples   : 450
  Test examples    : 80
  Effective batch  : 16
  Total steps est. : 1125
  Est. time        : ~2.3 min


In [18]:
print("Starting training...")
result = trainer.train()

print("\n" + "="*55)
print("TRAINING COMPLETE")
print("="*55)
print(f"  Runtime : {result.metrics['train_runtime']:.0f}s")
print(f"  Loss    : {result.metrics['train_loss']:.4f}")

trainer.save_model(MODEL_SAVE_DIR)
tokenizer.save_pretrained(MODEL_SAVE_DIR)
print(f"\n✓ Model saved → {MODEL_SAVE_DIR}")

Starting training...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,5.063226,4.915559,0.010451,0.007590,0.016771
2,4.575404,4.171499,0.000000,0.000000,0.000000
3,4.128767,3.894993,0.058005,0.064935,0.052411
4,3.718122,3.409880,0.062827,0.083624,0.050314
5,3.164640,2.938232,0.133333,0.162162,0.113208
6,2.621861,2.563407,0.243119,0.268354,0.222222
7,2.200844,2.183140,0.309129,0.305955,0.312369
8,1.846528,1.954589,0.380855,0.370297,0.392034
9,1.570654,1.740261,0.479769,0.443850,0.522013
10,1.291875,1.536933,0.566802,0.547945,0.587002


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


TRAINING COMPLETE
  Runtime : 389s
  Loss    : 1.0095


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✓ Model saved → /kaggle/working/vilegalbert-srl


In [19]:
from seqeval.metrics import classification_report as seq_report

print("Evaluating...")
eval_result = trainer.evaluate(tokenized_ds["test"])

print("\n" + "="*55)
print("TEST RESULTS")
print("="*55)
print(f"  Micro F1  : {eval_result['eval_f1']:.4f}")
print(f"  Precision : {eval_result['eval_precision']:.4f}")
print(f"  Recall    : {eval_result['eval_recall']:.4f}")
print(f"  Loss      : {eval_result['eval_loss']:.4f}")

preds_out   = trainer.predict(tokenized_ds["test"])
preds       = np.argmax(preds_out.predictions, axis=-1)
labels_out  = preds_out.label_ids

true_labels, true_preds = [], []
for pred_seq, label_seq in zip(preds, labels_out):
    true_labels.append([ID2LABEL[l] for p, l in zip(pred_seq, label_seq) if l != -100])
    true_preds.append( [ID2LABEL[p] for p, l in zip(pred_seq, label_seq) if l != -100])

print("\nPer-role breakdown:")
print(seq_report(true_labels, true_preds))

Evaluating...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



TEST RESULTS
  Micro F1  : 0.8112
  Precision : 0.7748
  Recall    : 0.8512
  Loss      : 1.0930

Per-role breakdown:
              precision    recall  f1-score   support

       Agent       0.91      1.00      0.95        31
   Condition       0.61      0.90      0.73        51
   Predicate       0.87      0.87      0.87       158
   Recipient       0.80      0.98      0.88        46
       Theme       0.73      0.76      0.75       186
        Time       0.71      1.00      0.83         5

   micro avg       0.77      0.85      0.81       477
   macro avg       0.77      0.92      0.84       477
weighted avg       0.78      0.85      0.81       477



In [20]:
import torch
import torch.distributed as dist
from transformers import AutoModelForTokenClassification, AutoTokenizer, pipeline

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

if dist.is_available() and dist.is_initialized():
    dist.destroy_process_group()
    print("✓ DDP destroyed")

torch.cuda.empty_cache()

infer_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_SAVE_DIR,
    torch_dtype = torch.float32,
)
infer_model.eval()

infer_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_SAVE_DIR,
    use_fast = False
)

srl_pipeline = pipeline(
    task                 = "ner",
    model                = infer_model,
    tokenizer            = infer_tokenizer,
    aggregation_strategy = "simple",
    device               = 0,
)

print(f"✓ SRL pipeline ready")
print(f"  dtype  : {next(infer_model.parameters()).dtype}")
print(f"  device : {next(infer_model.parameters()).device}")

# Sanity check
VALID_ROLES = set(ROLES)
demo = [
    "Bên A phải bàn giao vật tư cho bên B.",
    "Bên B phải thanh toán 10.000.000 VNĐ trước ngày 05/5/2024.",
    "Nếu thanh toán trễ hạn, mức phạt 1% mỗi ngày sẽ được áp dụng.",
]
print("\nSanity check:")
for sent in demo:
    spans = srl_pipeline(sent)
    print(f"\n  Clause: {sent}")
    for s in spans:
        if s["entity_group"] in VALID_ROLES and s["score"] >= 0.60:
            print(f"    [{s['entity_group']:<12}] '{s['word']}'  score={s['score']:.3f}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✓ SRL pipeline ready
  dtype  : torch.float32
  device : cuda:0

Sanity check:

  Clause: Bên A phải bàn giao vật tư cho bên B.
    [Agent       ] 'Bên'  score=0.691
    [Predicate   ] 'phải bàn giao'  score=0.941
    [Theme       ] 'vật tư'  score=0.961
    [Recipient   ] 'cho'  score=0.677
    [Recipient   ] 'bên B.'  score=0.926

  Clause: Bên B phải thanh toán 10.000.000 VNĐ trước ngày 05/5/2024.
    [Agent       ] 'B'  score=0.775
    [Predicate   ] 'phải thanh'  score=0.951
    [Theme       ] 'toán'  score=0.959
    [Theme       ] '10.000.000 VNĐ'  score=0.929
    [Time        ] 'trước ngày 05/@@'  score=0.637
    [Theme       ] '5/2024@@'  score=0.837

  Clause: Nếu thanh toán trễ hạn, mức phạt 1% mỗi ngày sẽ được áp dụng.
    [Condition   ] 'Nếu thanh toán trễ hạn@@'  score=0.957
    [Theme       ] 'mức phạt 1% mỗi ngày'  score=0.911
    [Predicate   ] 'sẽ'  score=0.922
    [Predicate   ] 'được áp dụ@@'  score=0.896


In [21]:
import time
import json

CONFIDENCE_THRESHOLD = 0.60
VALID_ROLES          = set(ROLES)
BATCH_SIZE           = 32
REPORT_EVERY         = 10_000

with open(CLAUSE_FILE, "r", encoding="utf-8") as f:
    all_clauses = [line.strip() for line in f if line.strip()]

print(f"✓ Reloaded {len(all_clauses):,} clauses")

t_start         = time.time()
total_processed = 0
total_with_srl  = 0
final_results   = []

for batch_start in range(0, len(all_clauses), BATCH_SIZE):
    batch = [c for c in all_clauses[batch_start : batch_start + BATCH_SIZE] if c.strip()]
    if not batch:
        continue

    try:
        results = srl_pipeline(batch)
    except Exception as ex:
        print(f"  ⚠ Skipping batch {batch_start}: {ex}")
        total_processed += len(batch)
        continue

    for clause, spans in zip(batch, results):
        predicate = None
        roles     = {}

        for s in spans:
            role  = s["entity_group"]
            score = s["score"]
            text  = s["word"].strip()

            if role not in VALID_ROLES or score < CONFIDENCE_THRESHOLD or not text:
                continue
                
            if role == "Predicate":
                if predicate is None:
                    predicate = text
                else:
                    predicate += f" {text}" 
            else:
                if role in roles:
                    roles[role] += f", {text}" 
                else:
                    roles[role] = text

        if not predicate or not roles:
            total_processed += 1
            continue

        # Format exact to specification
        record = {
            "clause"   : clause,
            "predicate": predicate,
            "roles"    : roles,
        }
        
        final_results.append(record)
        total_processed += 1
        total_with_srl  += 1

    if total_processed % REPORT_EVERY < BATCH_SIZE:
        elapsed = time.time() - t_start
        speed   = total_processed / max(elapsed, 1)
        eta     = (len(all_clauses) - total_processed) / max(speed, 1)
        print(f"  [{total_processed:>9,} / {len(all_clauses):,}]"
              f"  speed={speed:.0f}/s  ETA={eta/60:.1f}min")

with open(SRL_OUTPUT, "w", encoding="utf-8") as f_out:
    json.dump(final_results, f_out, ensure_ascii=False, indent=2)

elapsed = time.time() - t_start
print(f"\n✓ Saved: {SRL_OUTPUT}")
print(f"  Total processed        : {total_processed:,}")
print(f"  Clauses with SRL output: {total_with_srl:,} "
      f"({100*total_with_srl/max(total_processed,1):.1f}%)")
print(f"  Total time             : {elapsed:.0f}s = {elapsed/60:.1f}min")

✓ Reloaded 1,147,999 clauses


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  [   10,016 / 1,147,999]  speed=77/s  ETA=246.4min
  [   20,000 / 1,147,999]  speed=77/s  ETA=243.2min
  [   30,016 / 1,147,999]  speed=77/s  ETA=240.6min
  [   40,000 / 1,147,999]  speed=78/s  ETA=238.3min
  [   50,016 / 1,147,999]  speed=77/s  ETA=236.2min
  [   60,000 / 1,147,999]  speed=77/s  ETA=234.6min
  [   70,016 / 1,147,999]  speed=77/s  ETA=232.8min
  [   80,000 / 1,147,999]  speed=77/s  ETA=230.8min
  [   90,016 / 1,147,999]  speed=77/s  ETA=228.8min
  [  100,000 / 1,147,999]  speed=77/s  ETA=226.6min
  [  110,016 / 1,147,999]  speed=77/s  ETA=224.2min
  [  120,000 / 1,147,999]  speed=77/s  ETA=221.8min
  [  130,016 / 1,147,999]  speed=77/s  ETA=219.5min
  [  140,000 / 1,147,999]  speed=77/s  ETA=217.2min
  [  150,016 / 1,147,999]  speed=77/s  ETA=214.9min
  [  160,000 / 1,147,999]  speed=77/s  ETA=212.6min
  [  170,016 / 1,147,999]  speed=78/s  ETA=210.3min
  [  180,000 / 1,147,999]  speed=78/s  ETA=208.0min
  [  190,016 / 1,147,999]  speed=78/s  ETA=205.7min
  [  200,000